In [ ]:
import pandas as pd
import spacy
import spacy_syllables
import glob
import os
from statistics import mean
from num2words import num2words

full text analysis (boxplots are created in `complexity_boxplots.py`).
corresponding to thesis section "General language complexity"

In [ ]:
nlp = spacy.load("nl_core_news_md")
nlp.add_pipe("syllables", after="tagger", config={"lang": "nl"})


def is_passive(sentence):
    for token in sentence:
        if token.dep_ == "nsubj:pass":
            return True
    return False

def analyze_text(text):
    doc = nlp(text.lower())
    
    words = []
    total_syllables = 0

    for token in doc:
        if token.is_alpha:
            words.append(token.text)
            if token._.syllables_count is not None:
                total_syllables += token._.syllables_count

        elif token.is_digit:
            num_doc = nlp(num2words(int(token.text), lang="nl"))
            for nt in num_doc:
                if nt.is_alpha:
                    words.append(nt.text)
                    if nt._.syllables_count is not None:
                        total_syllables += nt._.syllables_count

    n_words = len(words)
    ttr = len(set(words)) / n_words if n_words else 0

    sentences = list(doc.sents)
    n_sentences = len(sentences)

    avg_sent_len = n_words / n_sentences if n_sentences else 0
    avg_syllables = total_syllables / n_words if n_words else 0

    fres = 206.835 - (1.015 * avg_sent_len) - (84.6 * avg_syllables)

    passive = sum(1 for sent in sentences if is_passive(sent))
    pass_pct = (passive / n_sentences) * 100 if n_sentences else 0

    return avg_sent_len, avg_syllables, fres, pass_pct, ttr

In [ ]:
level = []
folder_directory = "Abel_level"
for files in os.listdir(folder_directory):
    file_path = os.path.join(folder_directory, files)
    with open(file_path, 'r', encoding='utf-8') as f:
        file = f.read()
        df = pd.read_csv(file_path)
        turns = len(df) + 1 
        avg_sent_len, avg_syllables, fres, pass_pct, ttr = analyze_text(file)
        name = files.split("_Arts")[0].split("_Patient")[0]
        if "Arts" in str(file_path):
            label = "Doctor"
        else:
            label = "Patient"
        level.append({"filename":name, "label":label, "avg_sent_length":avg_sent_len, "avg_syllables":avg_syllables, "fres":fres, "pass_pct":pass_pct, "ttr":ttr})
df = pd.DataFrame(level)
df.to_csv("Level_Analysis/Abel_Level_Scores.csv")


Words and syllables (used for dialogue pace)

In [ ]:
import pandas as pd
import spacy
import spacy_syllables
import glob
import os
from statistics import mean
from num2words import num2words



nlp = spacy.load("nl_core_news_md")
nlp.add_pipe("syllables", after="tagger", config={"lang": "nl"})



def analyze_text(text):
    doc = nlp(text.lower())
    words = []
    for token in doc:
        if token.is_alpha:
            words.append(token)
        elif token.is_digit:
            num_doc = nlp(num2words(int(token.text), lang="nl"))
            words.extend([nt for nt in num_doc if nt.is_alpha])
    n_words = len(words)
    total_syllables = sum(t._.syllables_count for t in words if t._.syllables_count is not None)
    
    return n_words, total_syllables

level = []
folder_directory = "Abel_level"
for files in os.listdir(folder_directory):
    file_path = os.path.join(folder_directory, files)
    with open(file_path, 'r', encoding='utf-8') as f:
        file = f.read()
        df = pd.read_csv(file_path)
        turns = len(df) + 1 
        n_words, total_syllables = analyze_text(file)
        name = files.split("_Arts")[0].split("_Patient")[0]
        if "Arts" in str(file_path):
            label = "Doctor"
        else:
            label = "Patient"
        level.append({"filename":name, "label":label, "turns": turns, "n_words":n_words, "total_syllables":total_syllables})
df = pd.DataFrame(level)
df.to_csv("Level_Analysis/Ibl_words_syll.csv")

Change in language complexity (corresponding to thesis section "Language complexity adaptation")

In [ ]:
level = []
folder_directory = ["IBIS_level", "Abel_level", "IBIS _colon_level"]

for folder in folder_directory:
    for file_name in os.listdir(folder):
        file_path = os.path.join(folder, file_name)

        df = pd.read_csv(file_path)

        index1 = len(df) // 2

        p1 = " ".join(df.iloc[:index1]["text"].astype(str))
        p2 = " ".join(df.iloc[index1:]["text"].astype(str))

        
        # Analyze both halves
        metrics1 = analyze_text(p1)
        metrics2 = analyze_text(p2)
        
        # File name handling
        name = file_name.split("_Arts")[0].split("_Patient")[0]
        
        # Label handling
        label = "Doctor" if "Arts" in file_path else "Patient"
        
        # Append metrics for both halves in one row
        level.append({
            "filename": name,
            "label": label,
            "p1_fres": metrics1[2],
            "p2_fres": metrics2[2],
        })

# Save results
import pandas as pd
df_out = pd.DataFrame(level)


In [ ]:
df_out.to_csv("Level_Analysis/halves.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel

# Load previous output
df = pd.read_csv("Level_Analysis/all_half.csv")


# Pivot so doctor and patient are on the same row
pivot = df.pivot(index="filename", columns="label", values=["p1_fres", "p2_fres"])

# Flatten column names
pivot.columns = [
    f"{metric}_{label}" 
    for metric, label in pivot.columns
]

pivot = pivot.reset_index()

pivot

In [ ]:
pivot["delta_doctor"] = pivot["p2_fres_Doctor"] - pivot["p1_fres_Doctor"]
pivot["delta_patient"] = pivot["p2_fres_Patient"] - pivot["p1_fres_Patient"]

pivot["pct_doctor"] = pivot["delta_doctor"] / pivot["p1_fres_Doctor"] * 100
pivot["pct_patient"] = pivot["delta_patient"] / pivot["p1_fres_Patient"] * 100

pivot["distance_P1"] = abs(pivot["p1_fres_Doctor"] - pivot["p1_fres_Patient"])
pivot["distance_P2"] = abs(pivot["p2_fres_Doctor"] - pivot["p2_fres_Patient"])
pivot["converged"] = pivot["distance_P2"] < pivot["distance_P1"]
pivot["diverged"] = pivot["distance_P2"] > pivot["distance_P1"]

pivot

In [ ]:
def direction_stats(mask, pct_series):
    return (
        mask.mean() * 100,
        pct_series[mask].abs().mean(),
        pct_series[mask].abs().std()
    )

doc_harder = pivot["delta_doctor"] < 0
doc_easier = pivot["delta_doctor"] > 0
pat_harder = pivot["delta_patient"] < 0
pat_easier = pivot["delta_patient"] > 0

doc_harder_stats = direction_stats(doc_harder, pivot["pct_doctor"])
doc_easier_stats = direction_stats(doc_easier, pivot["pct_doctor"])
pat_harder_stats = direction_stats(pat_harder, pivot["pct_patient"])
pat_easier_stats = direction_stats(pat_easier, pivot["pct_patient"])

converged_pct = pivot["converged"].mean() * 100
diverged_pct = 100 - converged_pct

In [ ]:
doctor_converged_mean = pivot.loc[pivot["converged"], "pct_doctor"].mean()
patient_converged_mean = pivot.loc[pivot["converged"], "pct_patient"].mean()
doctor_diverged_mean = pivot.loc[pivot["diverged"], "pct_doctor"].mean()
patient_diverged_mean = pivot.loc[pivot["diverged"], "pct_patient"].mean()

doctor_overall_mean = pivot["pct_doctor"].mean()
patient_overall_mean = pivot["pct_patient"].mean()

In [ ]:
print("Doctor Harder:", doc_harder_stats)
print("Doctor Easier:", doc_easier_stats)
print("Patient Harder:", pat_harder_stats)
print("Patient Easier:", pat_easier_stats)
print("Converged %:", converged_pct)
print("Diverged %:", diverged_pct)
print("Doctor converged mean %:", doctor_converged_mean)
print("Patient converged mean %:", patient_converged_mean)
print("Doctor diverged mean %:", doctor_diverged_mean)
print("Patient diverged mean %:", patient_diverged_mean)
print("Doctor overall mean %:", doctor_overall_mean)
print("Patient overall mean %:", patient_overall_mean)